In [1]:
import simulator
from NoisyCircuits.utils.CreateNoiseModel import CreateNoiseModel
import numpy as np
import time
import pennylane as qml

2026-05-18 17:44:50,751	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
import pickle

with open("test_model.pkl", "rb") as f:
    data = pickle.load(f)

single = data["s"]
double = data["d"]

In [3]:
def build_random_unitary(n):
    N = 1 << n
    random_matrix = np.random.rand(N, N) + 1j * np.random.rand(N, N)
    Q, R = np.linalg.qr(random_matrix)
    D = np.diag(R)
    D = D / np.abs(D)
    return Q @ np.diag(D)

In [4]:
import random

def generate_random_qubit_list(n, b, a=0):
    return random.sample(range(a, b), n)

In [5]:
def test_unitary(U, apply_qubits, dev):
    @qml.qnode(dev)
    def apply_unitary():
        qml.QubitUnitary(U, wires=apply_qubits)
        return qml.state()
    return apply_unitary()

In [11]:
q = 12
# dev = qml.device("lightning.qubit", wires=q)
# unitary = build_random_unitary(q)
# apply_qubits = generate_random_qubit_list(q, q)
# instruction_lst = [["unitary", apply_qubits, unitary]]
t0 = time.perf_counter_ns()
state_pennylane = test_unitary(unitary, apply_qubits, dev)
state_custom = np.zeros(1 << q, dtype=np.complex128)
state_custom[0] = 1.0
t1 = time.perf_counter_ns()
simulator.simulate_circuit(instruction_lst, state_custom, single, double, q, False, 1, True, 4)
t2 = time.perf_counter_ns()
state_custom = state_custom.reshape([2]*q).transpose(list(range(q))[::-1]).reshape(-1)
print("States Match: ", np.allclose(state_custom, state_pennylane, atol=1e-10))
print(f"Time taken by Pennylane: {(t1 - t0) / 1e6:.2f} ms")
print(f"Time taken by Custom Simulator: {(t2 - t1) / 1e6:.2f} ms")

States Match:  True
Time taken by Pennylane: 301.94 ms
Time taken by Custom Simulator: 580.37 ms
